# 🌊 The Bayesian Neural ODE: Uncertainty in Continuous Time
Shifting from a deterministic Neural ODE to a **Bayesian Neural ODE (B-NODE)** is exactly how we solve the problem of the "single fixed curve." 

When dealing with chaotic or stiff dynamics (like a neuron firing), you do not just want the most likely path; you want to know *how much you can trust* that path.

---

## 1. The Theoretical Foundation: Why Certainty is Dangerous
A standard Neural ODE finds a single weight vector $\theta^*$ that minimizes the error between its predicted trajectory and your data. 

However, if your data is noisy, sparse, or missing chunks of time, there might be hundreds of different weight vectors that all fit the data equally well. The deterministic model arbitrarily picks one, drawing a highly confident line through the empty spaces. If it guesses the physics wrong in those gaps, it fails silently.

## 2. The Probabilistic Shift: An Ensemble of Curves
A Bayesian Neural ODE changes the fundamental question. Instead of asking, *"What is the best weight vector $\theta$?"* it asks, *"What is the probability distribution of all valid weight vectors given the data?"*

Mathematically, we want to find the posterior distribution $P(\theta \mid \mathcal{D})$. Assuming Gaussian errors, the likelihood of a weight vector $\theta$ being correct is proportional to how well it fits the data:
$$P(\theta) \propto e^{-\frac{1}{\sigma^2} \sum \|\text{Solve}(\theta) - \text{Data}\|^2}$$

By drawing multiple samples from this probability distribution, we generate an **ensemble of valid curves** rather than just one.

## 3. The Implementation Trick: MC-Dropout
Exactly computing $P(\theta)$ and running an ODE solver through it using traditional methods (like Hamiltonian Monte Carlo) takes a colossal amount of compute time.

Instead, we use **Monte Carlo Dropout (MC-Dropout)** as a variational approximation. We inject `Lux.Dropout` layers directly into the continuous-time Multi-Layer Perceptron (the $f(h, t, \theta)$). This transforms the continuous-time derivative function from a deterministic mapper into a probabilistic one.

## 4. The Continuous-Time Nuance
There is a critical technical nuance when applying dropout to ODEs: **You cannot randomly drop out neurons at every microsecond step the ODE solver takes.** 

If the architecture changes mid-step, it makes the derivative mathematically discontinuous and will immediately crash the ODE solver. 

Instead, we sample **one random dropout mask** (representing one specific sub-network) and keep it fixed for the entire forward integration across time. 
*   **Sub-network A** solves the ODE and draws one valid curve.
*   **Sub-network B** solves the ODE and draws a slightly different valid curve.

## 5. Adjoint Loss and Ensemble Optimization
The loss function remains pure data-matching. We use the memory-efficient **Adjoint Method** to backpropagate through the ODE solver.

Because the dropout mask is active during the forward pass, the optimizer is no longer optimizing a single network. The network is forced to learn robust continuous dynamics that work across an *entire ensemble* of sub-networks, creating a highly resilient model.

## 6. Monte Carlo Inference and Trajectory Uncertainty
To visualize what the Bayesian Neural ODE has learned, we stop predicting a single trajectory. Instead, we solve the ODE **50 different times**.

Crucially, before each ODE solve, we pass a dummy state vector through the network simply to trigger the random number generator, ensuring a completely new random dropout mask is applied to the network state (`st_infer`).

By calculating the variance across these 50 passes, we map out the exact boundaries of our **epistemic uncertainty**:
*   **Solid Line:** The Ensemble Mean (the most likely trajectory).
*   **Shaded Ribbon:** The 95% Confidence Interval. 

When the ribbon expands, the B-NODE is visually warning you that the continuous dynamics in that specific temporal region are highly uncertain.

In [ ]:
# %% Cell 1: Setup and Data Prep
using Lux, DifferentialEquations, SciMLSensitivity, Optimization, OptimizationOptimisers, Statistics, Random, ComponentArrays, Zygote, Plots

# Assuming t_train (shape: N) and z_train (shape: 4 x N) are available
# Example:
# t_train = Float32.(df_ordered.timestamp)
# z_train = Float32.(Matrix(df_ordered[:, [:V, :n, :m, :h]])')

u0 = z_train[:, 1] # Initial condition for the ODE solver
tspan = (t_train[1], t_train[end])

rng = Random.default_rng()
Random.seed!(rng, 42)

In [ ]:
# %% Cell 2: Bayesian MLP Architecture

# 5% dropout is usually sufficient for continuous-time models
dropout_rate = 0.05f0 

nn_bnode = Lux.Chain(
    Lux.Dense(4 => 32, tanh),
    Lux.Dropout(dropout_rate), # Injects epistemic uncertainty
    Lux.Dense(32 => 32, tanh),
    Lux.Dropout(dropout_rate), 
    Lux.Dense(32 => 4) 
)

# Initialize network parameters and the state (which holds the random dropout masks)
ps, st = Lux.setup(rng, nn_bnode)
p_initial = ComponentArray(ps)

In [ ]:
# %% Cell 3: Probabilistic Neural Dynamics

function bayesian_dynamics(u, p, t)
    # The network predicts du/dt based on the current dropout mask captured in `st`
    return nn_bnode(u, p, st)[1]
end

# Define the ODE Problem
bnode_prob = ODEProblem(bayesian_dynamics, u0, tspan, p_initial)

In [ ]:
# %% Cell 4: Loss Function and Training via Adjoint Method

function predict_trajectory(θ)
    _prob = remake(bnode_prob, p=θ)
    
    # Solve using Tsit5 and the Adjoint method for memory-efficient backpropagation
    sol = solve(_prob, Tsit5(), 
                saveat=t_train, 
                abstol=1e-6, reltol=1e-6,
                sensealg=InterpolatingAdjoint(autojacvec=ZygoteVJP()))
    return sol
end

function data_loss_bayesian(θ, _)
    sol = predict_trajectory(θ)
    
    if sol.retcode != ReturnCode.Success
        return Inf 
    end
    
    # Calculate MSE on the current randomly dropped-out subnetwork
    mse_loss = mean(abs2, Array(sol) .- z_train)
    return mse_loss
end

loss_history = Float32[]

callback_fn = function (θ, loss_val)
    push!(loss_history, loss_val)
    if length(loss_history) % 20 == 0
        println("Epoch $(length(loss_history)) | Ensemble Loss: $(round(loss_val, digits=5))")
    end
    return false
end

optf = OptimizationFunction(data_loss_bayesian, Optimization.AutoZygote())
optprob = OptimizationProblem(optf, p_initial)

println("Starting Bayesian Neural ODE Training...")
result = solve(optprob, Adam(0.005), maxiters = 1000, callback = callback_fn)
println("Training Complete!")

p_opt = result.u

In [ ]:
# %% Cell 5: Extracting the Trajectory Uncertainty Map
using Plots

num_samples = 50
predictions = zeros(Float32, 4, length(t_train), num_samples)

println("Running $num_samples Bayesian ODE trajectory solves...")

for i in 1:num_samples
    global st
    # 1. Trigger a state update to get a new random dropout mask.
    # We pass dummy data (u0) through the network simply to advance the RNG.
    _, st = nn_bnode(u0, p_opt, st)
    
    # 2. Solve the ODE trajectory using this newly generated sub-network mask
    sol = predict_trajectory(p_opt)
    
    # 3. Store the full continuous trajectory
    predictions[:, :, i] = Array(sol)
end

# Calculate the Empirical Mean (Most likely trajectory)
mean_trajectory = dropdims(mean(predictions, dims=3), dims=3)

# Calculate the Epistemic Uncertainty (Standard Deviation across the ensemble)
std_trajectory = dropdims(std(predictions, dims=3), dims=3)

# --- Plotting the Uncertainty Ribbon for Voltage ---
v_mean = mean_trajectory[1, :]
v_std = std_trajectory[1, :]

plot(t_train, v_mean, 
    ribbon = 2 .* v_std,  # Plot ±2 Standard Deviations (approx. 95% confidence)
    fillalpha = 0.3, 
    color = :cyan,
    linewidth = 2,
    title = "Bayesian Neural ODE: Voltage Trajectory Uncertainty",
    xlabel = "Time (ms)",
    ylabel = "Voltage (mV)",
    label = "Ensemble Mean (V)",
    grid = true
)

scatter!(t_train, z_train[1, :], 
    label="True Data", 
    markersize=3, 
    color=:orange,
    alpha=0.6
)